In [1]:
from robotblockset.tools import get_rbs_path, print_xml, print_xml_for_console, find_attr_values_in_xml

import mujoco 
from robotblockset.mujoco.tools_mjcf import print_body_tree, actuators_for_joint, replace_in_mjcf_file

In [2]:
ROBOT = "panda"
GRIPPER = None
CAMERA = None

In [3]:
MODEL_PATH = get_rbs_path() + "/mujoco/mjcf_models/"
print(MODEL_PATH)

d:\Leon\Python\RBS\robotblockset/mujoco/mjcf_models/


In [4]:
scene=mujoco.MjSpec.from_file(MODEL_PATH + "scene.xml")
scene.copy_during_attach = True
# print_xml(scene.to_xml())

In [5]:
type(scene)

mujoco._specs.MjSpec

In [6]:
robot_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{ROBOT}.xml")
robot_spec.copy_during_attach = True
print_body_tree(robot_spec.worldbody, robot_spec)
# print_xml(robot_spec.to_xml())

Body Tree for "world"
-base
--link0
---link1 (Joints: joint1-HINGE[Actuator: pos_joint1])
----link2 (Joints: joint2-HINGE[Actuator: pos_joint2])
-----link3 (Joints: joint3-HINGE[Actuator: pos_joint3])
------link4 (Joints: joint4-HINGE[Actuator: pos_joint4])
-------link5 (Joints: joint5-HINGE[Actuator: pos_joint5])
--------link6 (Joints: joint6-HINGE[Actuator: pos_joint6])
---------link7 (Joints: joint7-HINGE[Actuator: pos_joint7])
----------flange
-----------hand
------------left_finger (Joints: finger_joint1-SLIDE)
------------right_finger (Joints: finger_joint2-SLIDE)


In [7]:
attachment_frame = scene.worldbody.add_frame()
attachment_frame.attach_body(robot_spec.body("base"), f"{ROBOT}_")
scene.body(f"{ROBOT}_base").name = ROBOT

In [8]:
ori_val = f"{ROBOT}_base"
new_val = f"{ROBOT}"
for x in  find_attr_values_in_xml(scene.to_xml(), ori_val):
    if x[1] == "exclude" and "body" in x[3]:
        x[3] = x[3].replace("body", "bodyname")
    print(f"scene.{x[1]}('{x[2]}').{x[3]} = '{new_val}'")


In [9]:
# scene.body('hc20_base').name = 'hc20'
# scene.exclude('hc20_exclude_1').bodyname1 = 'hc20'

In [10]:
print_body_tree(scene.worldbody, scene)
# print_xml(scene.to_xml())

Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_hand
------------panda_left_finger (Joints: panda_finger_joint1-SLIDE)
------------panda_right_finger (Joints: panda_finger_joint2-SLIDE)


In [11]:
if GRIPPER is not None:
    gripper_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{GRIPPER}.xml")
    gripper_spec.copy_during_attach = True
    attachment_frame = scene.body(f"{ROBOT}_flange").add_frame(pos=[0, 0, 0], quat=[0.9238795, 0, 0, -0.3826834])
    attachment_frame.attach_body(gripper_spec.body("hand"), f'{ROBOT}_tool_')
    scene.delete(scene.site(f"{ROBOT}_TCP"))
    scene.site(f"{ROBOT}_tool_hand_TCP").name = f"{ROBOT}_TCP"

In [12]:
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_hand
------------panda_left_finger (Joints: panda_finger_joint1-SLIDE)
------------panda_right_finger (Joints: panda_finger_joint2-SLIDE)


In [13]:
for s in scene.sites:
    if s.name != "":
        print(s.name)

Target
panda_x
panda_y
panda_z
panda_flange
panda_TCP
panda_left_finger
panda_right_finger


In [14]:
for actuator in scene.actuators:
  print(actuator.name)

panda_pos_joint1
panda_pos_joint2
panda_pos_joint3
panda_pos_joint4
panda_pos_joint5
panda_pos_joint6
panda_pos_joint7
panda_gripper


In [15]:
if CAMERA is not None:
    object_spec = mujoco.MjSpec.from_file(MODEL_PATH + f"{CAMERA}.xml")
    object_spec.copy_during_attach = True
    attachment_frame = scene.worldbody.add_frame(pos=[1.0, 0.2, 0.6])
    attachment_frame.attach_body(object_spec.body("cam"), "RS_")

In [16]:
print_body_tree(scene.worldbody, scene)

Body Tree for "world"
-Target
-panda
--panda_link0
---panda_link1 (Joints: panda_joint1-HINGE[Actuator: panda_pos_joint1])
----panda_link2 (Joints: panda_joint2-HINGE[Actuator: panda_pos_joint2])
-----panda_link3 (Joints: panda_joint3-HINGE[Actuator: panda_pos_joint3])
------panda_link4 (Joints: panda_joint4-HINGE[Actuator: panda_pos_joint4])
-------panda_link5 (Joints: panda_joint5-HINGE[Actuator: panda_pos_joint5])
--------panda_link6 (Joints: panda_joint6-HINGE[Actuator: panda_pos_joint6])
---------panda_link7 (Joints: panda_joint7-HINGE[Actuator: panda_pos_joint7])
----------panda_flange
-----------panda_hand
------------panda_left_finger (Joints: panda_finger_joint1-SLIDE)
------------panda_right_finger (Joints: panda_finger_joint2-SLIDE)


In [17]:
# print_xml(scene.to_xml())

In [18]:
if GRIPPER is None:
    scene.modelname = f"{ROBOT}_scene"
else:
    scene.modelname = f"{ROBOT}_{GRIPPER}_scene"
scene.option.timestep = 0.001
with open(MODEL_PATH + f"{ROBOT}_scene.xml", "w") as f:
    f.write(scene.to_xml())

## Scene and robot definition verification

In [19]:
from robotblockset.mujoco.robots_pymujoco import *
scene = mujoco_scene(MODEL_PATH + f"{ROBOT}_scene.xml", show_camera=None)

In [20]:
r=eval(f"{ROBOT}(scene=scene, JointNames='auto')")

[RBS_INFO] [1773476082.781757593] [panda_PyMuJoCo]: Robot connected to MuJoCo


In [21]:
r.JMove(r.q_home)
print(r.x)

[ 0.3067 -0.0000  0.4897 -0.0002 -1.0000 -0.0035  0.0046]


In [22]:
print(r.Kinmodel()[0])

[ 0.3067 -0.0000  0.4897 -0.0002 -1.0000 -0.0035  0.0046]
